## 1. Mount Google Drive
Access the OpenNeuro EEG dataset stored in Google Drive.

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Extract OpenNeuro Dataset
Unzip the dataset so the EEG files can be accessed in Colab.

In [7]:
!unzip -q /content/drive/MyDrive/openneuro_data.zip -d /content/

In [13]:
!pip install mne scipy pandas matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 46.2 MB/s eta 0:00:00


In [10]:
import os

print(os.listdir("/content/Senior Design/sub-072"))

['eeg']


In [11]:
print(os.listdir("/content/Senior Design/sub-072/eeg"))

['sub-072_task-eyesclosed_eeg.json', 'sub-072_task-eyesclosed_eeg.set', 'sub-072_task-eyesclosed_channels.tsv']


In [14]:
import mne

file_path = "/content/Senior Design/sub-072/eeg/sub-072_task-eyesclosed_eeg.set"

raw = mne.io.read_raw_eeglab(file_path, preload=True)

print(raw)

<RawEEGLAB | sub-072_task-eyesclosed_eeg.set, 19 x 330550 (661.1 s), ~47.9 MiB, data loaded>


In [15]:
import mne
import numpy as np

file_path = "/content/Senior Design/sub-072/eeg/sub-072_task-eyesclosed_eeg.set"
raw = mne.io.read_raw_eeglab(file_path, preload=True)

def band_power(raw, low, high):
    band_raw = raw.copy().filter(low, high)
    band_data = band_raw.get_data()
    return np.mean(band_data ** 2)

delta_power = band_power(raw, 0.5, 4)
theta_power = band_power(raw, 4, 8)
alpha_power = band_power(raw, 8, 12)
beta_power  = band_power(raw, 12, 30)

print("Delta:", delta_power)
print("Theta:", theta_power)
print("Alpha:", alpha_power)
print("Beta:", beta_power)

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 4 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 4.00 Hz
- Upper transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 5.00 Hz)
- Filter length: 3301 samples (6.602 s)

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 4 - 8 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 4.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 3.00 Hz)
- Upper passband edge: 8.0

In [16]:
import pandas as pd

participants = pd.read_csv("/content/Senior Design/participants.tsv", sep="\t")
print(participants.head())
print(participants.columns)

  participant_id Gender  Age Group  MMSE
0        sub-001      F   57     A    16
1        sub-002      F   78     A    22
2        sub-003      M   70     A    14
3        sub-004      F   67     A    20
4        sub-005      M   70     A    22
Index(['participant_id', 'Gender', 'Age', 'Group', 'MMSE'], dtype='object')


## Feature Extraction from EEG Data

In [17]:
import os
import mne
import numpy as np
import pandas as pd

participants = pd.read_csv("/content/Senior Design/participants.tsv", sep="\t")

def band_power(raw, low, high):
    band_raw = raw.copy().filter(low, high, verbose=False)
    band_data = band_raw.get_data()
    return np.mean(band_data ** 2)

rows = []

for _, row in participants.iterrows():
    subject = row["participant_id"]
    label = row["Group"]

    file_path = f"/content/Senior Design/{subject}/eeg/{subject}_task-eyesclosed_eeg.set"

    if not os.path.exists(file_path):
        print(f"Missing file for {subject}")
        continue

    try:
        raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)

        delta = band_power(raw, 0.5, 4)
        theta = band_power(raw, 4, 8)
        alpha = band_power(raw, 8, 12)
        beta = band_power(raw, 12, 30)

        rows.append({
            "participant_id": subject,
            "delta": delta,
            "theta": theta,
            "alpha": alpha,
            "beta": beta,
            "label": label
        })

        print(f"Done: {subject}")

    except Exception as e:
        print(f"Error with {subject}: {e}")

features_df = pd.DataFrame(rows)


features_df.head()

Done: sub-001
Done: sub-002
Done: sub-003
Done: sub-004
Done: sub-005
Done: sub-006
Done: sub-007
Done: sub-008
Done: sub-009
Done: sub-010
Done: sub-011
Done: sub-012
Done: sub-013
Done: sub-014
Done: sub-015
Done: sub-016
Done: sub-017
Done: sub-018
Done: sub-019
Done: sub-020
Done: sub-021
Done: sub-022
Done: sub-023
Done: sub-024
Done: sub-025
Done: sub-026
Done: sub-027
Done: sub-028
Done: sub-029
Done: sub-030
Done: sub-031
Done: sub-032
Done: sub-033
Done: sub-034
Done: sub-035
Done: sub-036
Done: sub-037
Done: sub-038
Done: sub-039
Done: sub-040
Done: sub-041
Done: sub-042
Done: sub-043
Done: sub-044
Done: sub-045
Done: sub-046
Done: sub-047
Done: sub-048
Done: sub-049
Done: sub-050
Done: sub-051
Done: sub-052
Done: sub-053
Done: sub-054
Done: sub-055
Done: sub-056
Done: sub-057
Done: sub-058
Done: sub-059
Done: sub-060
Done: sub-061
Done: sub-062
Done: sub-063
Done: sub-064
Done: sub-065
Done: sub-066
Done: sub-067
Done: sub-068
Done: sub-069
Done: sub-070
Done: sub-071
Done: 

,participant_id,delta,theta,alpha,beta,label
0,sub-001,1.076390e-09,6.990714e-11,1.646727e-11,1.527051e-11,A
1,sub-002,9.481848e-10,7.309760e-11,3.820053e-11,1.811247e-11,A
2,sub-003,7.689855e-10,8.577101e-11,4.954642e-11,1.530622e-11,A
3,sub-004,1.226396e-09,6.799503e-11,1.683207e-11,1.977086e-11,A
4,sub-005,1.325153e-09,7.100719e-11,2.029368e-11,2.104239e-11,A


## A = Alzheimer's C = Control F = FTD

In [18]:
print(features_df.shape)
print(features_df.head())
print(features_df["label"].value_counts())

(84, 6)
  participant_id         delta         theta         alpha          beta label
0        sub-001  1.076390e-09  6.990714e-11  1.646727e-11  1.527051e-11     A
1        sub-002  9.481848e-10  7.309760e-11  3.820053e-11  1.811247e-11     A
2        sub-003  7.689855e-10  8.577101e-11  4.954642e-11  1.530622e-11     A
3        sub-004  1.226396e-09  6.799503e-11  1.683207e-11  1.977086e-11     A
4        sub-005  1.325153e-09  7.100719e-11  2.029368e-11  2.104239e-11     A
label
A    36
C    29
F    19
Name: count, dtype: int64


## Random Forest Classification Results

In [19]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder


X = features_df[["delta", "theta", "alpha", "beta"]]


y = features_df["label"]


le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Label mapping:")
for original, encoded in zip(le.classes_, range(len(le.classes_))):
    print(f"{original} -> {encoded}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)


model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)


y_pred = model.predict(X_test)


print("\nAccuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

Label mapping:
A -> 0
C -> 1
F -> 2

Accuracy: 0.5882352941176471

Confusion Matrix:
[[6 1 0]
 [2 3 1]
 [2 1 1]]

Classification Report:
              precision    recall  f1-score   support

           A       0.60      0.86      0.71         7
           C       0.60      0.50      0.55         6
           F       0.50      0.25      0.33         4

    accuracy                           0.59        17
   macro avg       0.57      0.54      0.53        17
weighted avg       0.58      0.59      0.56        17

